# ifuCube: the Helix Nebula, end to end

A walk through the package on a real field: read an LVM DAP file, build
emission-line maps, check that no flux was lost, make a Balmer-decrement map,
build a cube, and look at the fibres themselves.

NGC 7293 is a good first target and a bad one. Good, because it is bright,
nearby and resolved, so everything is visible. Bad, because it is a planetary
nebula — a thin expanding shell, not the H II region this pipeline was written
to catalogue — so some of the numbers below are not what you would get on a
star-forming region. Where that matters it is flagged.

**What you need:** one LVM DAP file for a Helix pointing. Nothing here
downloads data; set the path in the next cell.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# The next section downloads the data and sets DAP and SFRAME. If you
# already have files locally, set them here instead and skip it.
DAP = None
SFRAME = None

# Grid parameters. The two that matter:
#
#   FWHM is set from the FIBRE SPACING (37"), not from the pixel size. A
#   kernel narrower than the spacing leaves holes between fibres however
#   fine you make the grid.
#
#   PIXEL at 9" gives about 4 pixels across the ~35" resolution element,
#   which is comfortably Nyquist. Finer costs memory squared and buys no
#   information: a 2" cube is ~20x the bytes of a 9" one.
PIXEL = 9.0          # arcsec
FWHM = 35.0          # arcsec
FIBER_RADIUS = 17.65  # arcsec  (= 35.3 / 2)

OUT = Path("mapcats")
OUT.mkdir(exist_ok=True)

print(f"pixel {PIXEL}\" | kernel FWHM {FWHM}\" | fibre radius {FIBER_RADIUS}\"")

## 2. Getting the data

The Helix is one of six regions in the LVM preview released with **SDSS
DR20**, and it is public — no account, no credentials.

The pointing analysed in the LVM Helix paper (Sánchez et al. 2026) is

| | |
|---|---|
| tile ID | 11111 |
| tile group | 0011XX (derived from the tile ID) |
| MJD | 60191 |
| exposure | 4297 |

Products live on the Science Archive Server under a path built from exactly
those numbers:

    .../spectro/lvm/redux/{drpver}/{tilegrp}/{tileid}/{mjd}/lvmSFrame-{expnum}.fits
    .../spectro/lvm/analysis/{drpver}/{dapver}/{tilegrp}/{tileid}/{mjd}/{expnum}/dap-{rspid}-{snlevel}-{expnum}.dap.fits

so for DR20 (`drpver = 1.2.0`, `dapver = 1.2.0.251218`) the DAP file is
`dap-rsp108-sn20-00004297.dap.fits`. Note the `.dap.fits` ending — there is a
companion `.model.fits` alongside it holding the fitted models.

Two ways to fetch it. Use the second if you do not want to install anything.

### Option A — `sdss_access`

This is the SDSS way, and it is worth knowing because it builds the paths for
you from the numbers above rather than from a URL you have to keep correct. It
derives `tilegrp` from `tileid` on its own.

In [ ]:
# pip install sdss-access
from sdss_access import Access

DRPVER = "1.2.0"
DAPVER = "1.2.0.251218"
HELIX = dict(tileid=11111, mjd=60191, expnum=4297)

access = Access(release="DR20")      # public=True automatically for a DR
print("public release:", access.public)

dap_kw = dict(drpver=DRPVER, dapver=DAPVER, rspid="rsp108",
              snlevel="sn20", daptype="dap", **HELIX)

# Where things will come from, and where they will land. Check these before
# downloading several GB.
print("\nDAP url :", access.url("lvm_dap", **dap_kw))
print("DAP file:", access.full("lvm_dap", **dap_kw))
print("\nSFrame url :", access.url("lvm_sframe", drpver=DRPVER, **HELIX))
print("SFrame file:", access.full("lvm_sframe", drpver=DRPVER, **HELIX))

In [ ]:
# Download. Files land under $SAS_BASE_DIR (default ~/sas) in the same tree
# as on the server, so access.full(...) is where they end up.
access.remote()
access.add("lvm_dap", **dap_kw)
access.add("lvm_sframe", drpver=DRPVER, **HELIX)
access.set_stream()
access.commit()

DAP = Path(access.full("lvm_dap", **dap_kw))
SFRAME = Path(access.full("lvm_sframe", drpver=DRPVER, **HELIX))
print("DAP   :", DAP, DAP.exists())
print("SFrame:", SFRAME, SFRAME.exists())

### Option B — just the URL

No extra package. `download_file` caches, so re-running costs nothing.

In [ ]:
from astropy.utils.data import download_file
import shutil

SAS = "https://data.sdss.org/sas/dr20/spectro/lvm"
urls = {
    "dap": f"{SAS}/analysis/1.2.0/1.2.0.251218/0011XX/11111/60191/00004297/"
           f"dap-rsp108-sn20-00004297.dap.fits",
    "sframe": f"{SAS}/redux/1.2.0/0011XX/11111/60191/lvmSFrame-00004297.fits",
}

# The DAP file is the smaller one; start with it.
cached = download_file(urls["dap"], cache=True, show_progress=True)
DAP = Path("dap-rsp108-sn20-00004297.dap.fits")
shutil.copy(cached, DAP)
print("\nwrote", DAP, f"({DAP.stat().st_size / 1e6:.0f} MB)")

### Finding other exposures yourself

Hard-coding exposure 4297 is fine for following along, but DR20 contains 169
reduced exposures and more than one may cover the Helix. The summary files are
the index: `drpall` lists every reduced exposure, `dapall` every analysed one.
Read one of those and select by position, and you are no longer relying on a
number copied out of a paper.

This is also the honest way to check what is actually in the release. The
exposure above is the one the Helix paper used, and that paper describes the
**DR19** version of this field; DR20 reprocessed everything with a new
pipeline version, so confirm the exposure is present rather than assuming it.

In [ ]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u

dapall_url = (f"{SAS}/analysis/1.2.0/1.2.0.251218/"
              f"dapall-1.2.0-1.2.0.251218.fits")
dapall = Table.read(download_file(dapall_url, cache=True))
print(f"{len(dapall)} analysed exposures in DR20")
print("columns:", dapall.colnames[:14], "...")

# NGC 7293
helix = SkyCoord(337.4108 * u.deg, -20.8372 * u.deg)

# Column naming varies between releases, so find the coordinate pair rather
# than assuming it.
racol = next((c for c in dapall.colnames if c.lower() in ("ra", "plug_ra", "ifura")), None)
deccol = next((c for c in dapall.colnames if c.lower() in ("dec", "plug_dec", "ifudec")), None)
print("using coordinate columns:", racol, deccol)

if racol and deccol:
    sep = helix.separation(SkyCoord(dapall[racol] * u.deg, dapall[deccol] * u.deg))
    near = dapall[sep < 1 * u.deg]
    near["sep_arcmin"] = sep[sep < 1 * u.deg].to(u.arcmin).round(2)
    print(f"\n{len(near)} exposures within 1 degree of the Helix:")
    cols = [c for c in ("tileid", "mjd", "expnum", "sep_arcmin") if c in near.colnames]
    print(near[cols])

Pick an `expnum` from that table and feed it back into Option A, and the rest
of the notebook runs on whichever exposure you chose.

## 3. Read the fibres

`read_dap_multi` returns a `FiberSet`: fluxes, inverse variances, wavelengths,
fibre positions, and a metadata dictionary carrying the per-fibre catalogue.

It works out which fibres are science fibres and which are healthy **from the
file itself**, rather than from an external fibre map that may have changed
since the exposure. Rejected fibres have their fluxes set to NaN rather than
being dropped, so the ordering stays aligned with the projection matrix, and
every fibre in the input is recorded as kept or rejected.

In [ ]:
from ifuCube.io_lvm import read_dap_multi, list_dap_columns

# What is actually in this file? Useful the first time, and when a DAP
# release renames something.
list_dap_columns(DAP)

In [ ]:
fib = read_dap_multi(DAP, slitmap_path=DAP, verbose=True)

print()
print("lines      :", fib.meta["line_names"])
print("fibres kept:", fib.flux.shape[0])
print("mask source:", fib.meta["mask_info"]["source"])

`slitmap_path=DAP` points the masking at the DAP file's own `PT` extension,
which carries a `mask` column. Pass a different file if you want the mask to
come from somewhere else.

## 4. Build the maps

Each fibre's flux is spread over nearby pixels with a Gaussian kernel, and all
the weights go into one sparse matrix `A` of shape (pixels, fibres). The
columns of `A` sum to exactly 1, so every fibre hands out all of its flux and
none of it is lost.

That single property is the design. It makes the map **flux-conserving**:
summing the pixels of a region gives the total flux in that region.

In [ ]:
from ifuCube.mapcat_build import build_map, save_map_to_fits

res = build_map(fib,
                pixel_size_arcsec=PIXEL,
                fwhm_arcsec=FWHM,
                fiber_radius_arcsec=FIBER_RADIUS)

print("grid      :", res["inside"].shape)
print("footprint :", int(res["inside"].sum()), "pixels")
print("A         :", res["A"].shape, f"({res['A'].nnz:,} non-zero)")

## 5. Check it before trusting it

Run this after any parameter change. It is cheap and it is the thing that
catches a mis-specified grid.

In [ ]:
from ifuCube.diagnostics import run_all_checks

run_all_checks(res)

Checks 1, 2 and 4 are the real tests: total flux out over in, the operator
against its adjoint, and every column of `A` summing to one.

**Check 3 is informational, not pass/fail.** The partition-of-unity ratio is a
neighbourhood average, which is exact only for a constant field, so its spread
measures structure in the *source*, not flux loss. On a flat field it is 0; on
a ring like the Helix, around 0.3. A wide spread here is the nebula, not a bug.

## 6. Coverage — the one that will catch you out

`coverage` is the row sum of `A`: how much kernel weight each pixel received.
Because the columns of `A` sum to 1,

$$\sum_p \mathrm{coverage}(p) = \sum_p \sum_k A_{pk} = \sum_k 1 = K$$

**exactly**, for any kernel width and any pixel size. So its mean over the
footprint is $K/N_\mathrm{inside}$ — around 0.19 for 15" pixels, not ~1. It is
not a fraction and has no fixed ceiling.

A threshold like `coverage > 0.3` therefore masks the entire map. Use
`coverage_norm`, which divides by that plateau and sits near 1 across the
well-sampled interior whatever the pixel size and kernel.

In [ ]:
cov, cn, inside = res["coverage"], res["coverage_norm"], res["inside"]

print(f"coverage      : {cov[inside].min():.4f} .. {cov[inside].max():.4f}"
      f"   (median {np.median(cov[inside]):.4f})")
print(f"cov_plateau   : {res['cov_plateau']:.4f}   = K / N_inside")
print(f"coverage_norm : median {np.median(cn[inside]):.4f}")
print()
print(f"`coverage > 0.3`      keeps {100 * (cov[inside] > 0.3).mean():5.1f}% of the footprint")
print(f"`coverage_norm > 0.3` keeps {100 * (cn[inside] > 0.3).mean():5.1f}%")

In [ ]:
from ifuCube.plotting import good_pixels

# The mask to actually use: inside the footprint and well sampled. 0.5 means
# half the typical interior sampling, at any pixel size.
mask = good_pixels(res, min_coverage=0.5)
print(f"good_pixels keeps {100 * mask.sum() / inside.sum():.0f}% of the footprint")

## 7. Look at it

Maps come out North up and East left, the standard sky orientation.

Index planes **by name**, never by position. Reorder the `lines` argument once
and positional indexing hands you an H-alpha map labelled H-beta, which looks
entirely plausible.

In [ ]:
from ifuCube.plotting import show_map, show_planes

i = {n: k for k, n in enumerate(res["line_names"])}

show_map(res["map"][i["Halpha"]], wcs=res["wcs2d"], mask=mask,
         title="Helix, H-alpha", label="flux / pixel", log=True)
plt.show()

In [ ]:
# Several lines at once, each with its own stretch
want = [n for n in ("Halpha", "Hbeta", "OIII5007", "NII6583") if n in i]
show_planes(np.stack([res["map"][i[n]] for n in want]), want,
            wcs=res["wcs2d"], mask=mask, ncols=2, log=True,
            label="flux / pixel")
plt.show()

## 8. Line ratios

Ratios are the easiest thing this package does, because **everything
cancels**. Both lines go through the same `A`, so

$$\frac{(Af_1)_p}{(Af_2)_p}$$

is already correct: coverage, pixel area and the aperture fill factor all
divide out algebraically. Take ratios straight off the stored planes — no
corrections, no normalisation.

What does *not* cancel is noise. H-beta is the faint line, and a ratio diverges
where its denominator falls into the noise, so cut on signal-to-noise in the
weak line rather than on geometry.

In [ ]:
ha, hb = res["map"][i["Halpha"]], res["map"][i["Hbeta"]]
iv_hb = res["ivar"][i["Hbeta"]]

snr_hb = np.where(iv_hb > 0, hb * np.sqrt(iv_hb), 0.0)
ok = mask & (snr_hb > 3) & (hb > 0)

ratio = np.where(ok, ha / hb, np.nan)
print(f"S/N(Hbeta) > 3 over {100 * ok.sum() / mask.sum():.0f}% of the good pixels")
print(f"Halpha/Hbeta: 16/50/84% = "
      f"{np.nanpercentile(ratio[ok], [16, 50, 84]).round(3)}")

In [ ]:
# Balmer decrement -> extinction. Case B recombination gives an intrinsic
# ratio of 2.86, and E(B-V) = 2.32 * log10(R / 2.86).
#
# For the Helix specifically, treat this loosely: a thin expanding shell is
# not the dusty H II region the 2.86 assumption is built for, and the
# Halpha/Hbeta ratio in a planetary nebula carries temperature and density
# information as well as dust.
ebv = np.where(ok, 2.32 * np.log10(np.clip(ratio, 1e-3, None) / 2.86), np.nan)

show_map(ratio, wcs=res["wcs2d"], title="Halpha / Hbeta",
         label="ratio", cmap="viridis", pct=(2, 98))
plt.show()

## 9. Kinematics use a different operator

Flux is **extensive** — it adds up, so it is deposited with column-normalised
weights and totals are preserved.

Velocity is **intensive** — it does not add up. Splatting it the same way would
give a pixel the *sum* of its neighbours' velocities, so a region uniformly at
-27 km/s with three contributing fibres would read -81 km/s. The right operator
is a weighted average over the same matrix:

$$v_p = \frac{\sum_k A_{pk}\,w_k\,v_k}{\sum_k A_{pk}\,w_k}$$

Note also how bad fibres are handled. `build_map` scrubs bad fluxes to 0.0,
which is harmless because zero flux contributes zero. Here it would be a
disaster: **0 km/s is a valid velocity**, so a dead fibre must have its
*weight* zeroed, not its value.

`build_kinematic_map` takes the `result` from `build_map` and reuses its `A`,
so the kinematic map lands on exactly the same grid and WCS.

In [ ]:
from ifuCube.mapcat_build import build_kinematic_map

kin = build_kinematic_map(res, quantity="vel", weighting="ivar")

v = kin["map"][0]
print(f"\nmedian velocity: {np.nanmedian(v[np.isfinite(v)]):+.1f} km/s")
print("NGC 7293 systemic velocity is about -27 km/s")

In [ ]:
show_map(v, wcs=res["wcs2d"], mask=mask & np.isfinite(v),
         title="Helix, H-alpha velocity", label="km/s",
         cmap="RdBu_r", pct=(5, 95))
plt.show()

`neff_kin` is the honest independent-sample count for these maps — the Kish
effective fibre count on the *same* weights that formed the average, not
`res["neff"]`.

A caution if you fit a gradient to this map: there is a known field-wide
velocity gradient in LVM data of order 30 km/s per 1000 arcsec whose origin is
not settled, and the three spectrograph blocks do not behave identically. Check
for a jump between blocks before claiming an expansion velocity.

## 10. Dispersion is in Angstrom, not km/s

The DAP `disp_*` columns are line widths in **Angstrom**. Convert with
$\sigma_v = c\,\sigma_\lambda/\lambda$, and subtract the instrumental width
in quadrature before calling anything physical.

In [ ]:
from astropy import constants as const

dmap = build_kinematic_map(res, quantity="disp", weighting="ivar")
d_ang = dmap["map"][0]

c_kms = const.c.to("km/s").value
lam_ha = 6562.85
sigma_v = c_kms * d_ang / lam_ha

# LVM resolution R ~ 4000 near Halpha -> sigma_inst ~ 0.70 A ~ 32 km/s
sigma_inst = c_kms * 0.70 / lam_ha
intrinsic = np.sqrt(np.clip(sigma_v**2 - sigma_inst**2, 0, None))

print(f"observed  sigma: {np.nanmedian(sigma_v[np.isfinite(sigma_v)]):.1f} km/s")
print(f"instrument      : {sigma_inst:.1f} km/s")
print(f"intrinsic       : {np.nanmedian(intrinsic[np.isfinite(intrinsic)]):.1f} km/s")
print("\nIf intrinsic ~ 0 the line is unresolved -- which is the usual")
print("answer at R ~ 4000 for anything but a fast outflow.")

## 11. Classify fibres, not pixels

For a BPT diagram or any statistics, use the **fibre catalogue**, not the map.

At 9" pixels a 35" kernel puts each fibre on roughly 15 pixels, so a per-pixel
scatter plot shows every measurement about 15 times over, with neighbouring
points correlated because they share fibres. Density contours, a KDE bandwidth
or a mixture model fitted to that are overconfident by roughly sqrt(15).

The map's job is to show you *where*. The catalogue has one independent row per
measurement.

In [ ]:
cat = res["fibercat"]
n_pix = int(mask.sum())
n_fib = len(cat["fiberid"])
print(f"{n_pix} good pixels from {n_fib} fibres "
      f"-> {n_pix / n_fib:.1f}x oversampled")
print(f"median neff per pixel: {np.nanmedian(res['neff'][mask]):.2f}")

In [ ]:
from ifuCube.plotting import show_fibers

show_fibers(cat, cat["vel_Halpha"], label="velocity [km/s]",
            cmap="RdBu_r", pct=(5, 95))
plt.title("the fibres themselves -- no resampling")
plt.show()

## 12. Save, and read back

Maps are stored as **flux per pixel**, the conserving quantity. `read_map`
converts to surface brightness on the way in, so values can be compared
between maps built at different pixel sizes.

`read_map` also gives you `index()`, so nothing downstream has to count
planes.

In [ ]:
from ifuCube.mapcat_build import read_map

save_map_to_fits(res, OUT / "helix_map.fits")
save_map_to_fits(kin, OUT / "helix_vel.fits")

m = read_map(OUT / "helix_map.fits")
print("units       :", m["bunit"])
print("lines       :", m["line_names"])
print("index(Hbeta):", m["index"]("Hbeta"))
print("FIBERCAT    :", len(m["fibercat"]), "rows")

## 13. A cube

Same operator, same geometry, but over every wavelength plane instead of a
handful of fitted lines. This needs the reduced spectra (`lvmSFrame`), not the
DAP file.

**Cubes are stored as surface brightness**, since those values do not depend on
the grid the cube was built with. Size scales as the pixel size *squared*: at
9" a full pointing is about 3.3 GB for flux and inverse variance together, at
2" about 66 GB.

`inplace=True` converts without allocating a second cube. Use it whenever
nothing reads `c["cube"]` afterwards — but not otherwise, or its values quietly
become surface brightness while your code still treats them as flux.

In [ ]:
from ifuCube.io_lvm import read_lvm_sframe
from ifuCube.cube_build import build_cube, save_cube_to_fits, read_cube

if SFRAME is not None and Path(SFRAME).exists():
    cfib = read_lvm_sframe(SFRAME, verbose=True)
    c = build_cube(cfib, pixel_size_arcsec=PIXEL, fwhm_arcsec=FWHM,
                   fiber_radius_arcsec=FIBER_RADIUS, chunk_mb=256.0)
    save_cube_to_fits(c, "cubes/helix_cube.fits", inplace=True)
    print("\nwrote cubes/helix_cube.fits")
else:
    print("no SFrame -- run the download section to get one, "
          "or skip the cube and carry on with maps")

Read a **window**, not the whole cube. `wave_range` memmaps and slices, so an
H-alpha window costs a few MB instead of gigabytes.

In [ ]:
if Path("cubes/helix_cube.fits").exists():
    from ifuCube.plotting import show_wave

    cc = read_cube("cubes/helix_cube.fits", wave_range=(6540, 6590))
    print("window shape:", cc["data"].shape, "|", cc["bunit"])

    # width averages over a few Angstrom -- a single plane is noisy
    show_wave("cubes/helix_cube.fits", [6548., 6563., 6583.], width=3.0,
              ncols=3, cmap="inferno")
    plt.show()

## 14. Filling holes — for the picture, not the measurement

Dead fibres leave gaps. `fill_missing_fibers` estimates them from their
neighbours, at the **fibre** level, before any projection.

It is worth being clear about what this buys. Measured against a complete map:

| | 5% dead | 15% dead |
|---|---|---|
| per-pixel line ratio, unfilled | off by ~0.02 | off by ~0.02 |
| aperture flux, unfilled | -7.6% | -23.7% |
| aperture flux, filled | +0.1% | -0.8% |
| aperture flux, divided by coverage | -0.9% | -1.9% |

**Ratios need no filling at all** — a dead fibre is dead in every line, so the
geometric deficit cancels. Aperture photometry *is* biased low when unfilled,
but dividing by coverage fixes it to 1–2% **without inventing any data**, which
is the better correction: a known geometric factor instead of an interpolation.

So fill for the figure, and keep the unfilled map for the measurement. Imputed
fibres are flagged in the catalogue so they can be excluded from anything
needing independent samples.

In [ ]:
from ifuCube.fill import fill_missing_fibers

# max_gap_spacings is the guard: a fibre whose neighbours average further
# than this away sits inside a contiguous dead patch, and gets refused
# rather than guessed.
filled = fill_missing_fibers(fib, method="idw", k=6,
                             max_gap_spacings=1.35, ivar_mode="loo")
info = filled.meta["fill_info"]
print(f"\nimputed {info['n_filled']}, refused {info['n_too_far']}")

res_f = build_map(filled, pixel_size_arcsec=PIXEL, fwhm_arcsec=FWHM,
                  fiber_radius_arcsec=FIBER_RADIUS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4),
                         subplot_kw=dict(projection=res["wcs2d"]))
show_map(res["map"][i["Halpha"]], wcs=res["wcs2d"], ax=axes[0], log=True,
         title="measured", label="flux / pixel")
show_map(res_f["map"][i["Halpha"]], wcs=res["wcs2d"], ax=axes[1], log=True,
         title="filled (display only)", label="flux / pixel")
plt.show()

## 15. Aperture photometry, done properly

Use the **unfilled** map and correct by coverage. Dividing locally compensates
for the fibres that are missing; multiplying by the plateau restores the
normalisation, so the result is still a flux and still additive.

In [ ]:
plateau = res["cov_plateau"]
ap = mask & (cn > 0.3)

total = np.nansum(ha[ap] / cov[ap]) * plateau
naive = np.nansum(ha[ap])

print(f"coverage-corrected total: {total:.4g}")
print(f"uncorrected total       : {naive:.4g}   ({100 * (naive / total - 1):+.1f}%)")

## 16. Where to go next

- `ifuCube.ratio` — ratio maps with propagated errors, written to FITS
- `ifuCube.regions` — pick fibres out by circle or polygon and coadd them
- `ifuCube.mosaic` — combine overlapping pointings onto a common grid
- `ifuCube.healpix_build` — move maps onto HEALPix, merge and plot

Both builders also run from the command line, which is what you want for a
batch of exposures:

    python -m ifuCube.mapcat_build --dap FILE -o mapcats/x.fits --pixel 9 \
        --kinematics vel disp --check
    python -m ifuCube.cube_build --sframe FILE -o cubes/x.fits --pixel 9

Add `--help` to either for the full list.